# LightGBM LambdaRank Model

Ranking model using LightGBM's `lambdarank` objective, which optimises NDCG directly.

In [1]:
import sys
from pathlib import Path

cwd = Path(".").resolve()
REPO_ROOT = cwd if (cwd / ".git").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

import json
from itertools import product

import numpy as np
import pandas as pd
import lightgbm as lgb

from src.utils import load_train, load_train_cached_sample, train_val_split, compute_ndcg_fast
from src.features import build_features
from src.svd_features import build_svd_features
from src.position_features import build_position_features
from src.modeling import get_lgb_device_params

In [2]:
USE_SAMPLE = False
SAMPLE_FRAC = 0.15
REFRESH_CACHE = False
DEVICE_PARAMS = get_lgb_device_params()

if USE_SAMPLE:
    df = load_train_cached_sample(sample_frac=SAMPLE_FRAC, refresh=REFRESH_CACHE)
    print(f"Loaded {SAMPLE_FRAC*100:.0f}% sample")
else:
    df = load_train()
    print("Loaded full training data")

n_cols_before = df.shape[1]
df = build_features(df)
print(f"Loaded {len(df):,} rows, {df['srch_id'].nunique():,} searches")
print(f"Columns: {n_cols_before} raw + {df.shape[1] - n_cols_before} engineered = {df.shape[1]} total")

nan_pct = df.isnull().mean()
print(f"\nColumns with NaNs:")
print((nan_pct[nan_pct > 0] * 100).round(1).to_string())

GPU mode enabled - using LightGBM GPU acceleration


Loaded full training data


/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["comp_cheaper_frac"] = (
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_missing"] = df[col].isnull().astype("int8")
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:114: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["month"] = dt.dt.month.astype("int8")
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:115: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["day_of_week"] = dt.dt.dayofweek.astype("int8")
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti

/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:120: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["booking_window_bucket"] = pd.cut(
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:135: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["score2ma"] = (
/home/akshitanchan/Projects/hotel-trivago-but-its-expedia/src/features.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  C

Loaded 4,958,347 rows, 199,795 searches
Columns: 54 raw + 64 engineered = 118 total



Columns with NaNs:
visitor_hist_starrating             94.9
visitor_hist_adr_usd                94.9
prop_review_score                    0.1
prop_location_score2                22.0
srch_query_affinity_score           93.6
orig_destination_distance           32.4
comp1_rate                          97.6
comp1_inv                           97.4
comp1_rate_percent_diff             98.1
comp2_rate                          59.2
comp2_inv                           57.0
comp2_rate_percent_diff             88.8
comp3_rate                          69.1
comp3_inv                           66.7
comp3_rate_percent_diff             90.5
comp4_rate                          93.8
comp4_inv                           93.1
comp4_rate_percent_diff             97.4
comp5_rate                          55.2
comp5_inv                           52.4
comp5_rate_percent_diff             83.0
comp6_rate                          95.2
comp6_inv                           94.7
comp6_rate_percent_diff             9

In [3]:
train_df, val_df = train_val_split(df)
print(f"Train: {train_df.shape[0]:,}  Val: {val_df.shape[0]:,}")

n_before = df.shape[1]
df = build_svd_features(train_df, df)
print(f"SVD features added: {df.shape[1] - n_before}")

n_before = df.shape[1]
df = build_position_features(train_df, df)
print(f"Position-debiased features added: {df.shape[1] - n_before}")

train_df, val_df = train_val_split(df)

def make_relevance(frame):
    """booking=5, click-only=1, nothing=0"""
    return (frame["booking_bool"] * 5 +
            (frame["click_bool"] - frame["booking_bool"]).clip(lower=0)).values

y_train = make_relevance(train_df)
y_val = make_relevance(val_df)

DROP_COLS = [
    "srch_id", "date_time", "position", "click_bool", "booking_bool",
    "gross_bookings_usd", "random_bool",
]
COMP_RAW = [f"comp{i}_{s}" for i in range(1, 9) for s in ("rate", "inv", "rate_percent_diff")]
drop = set(DROP_COLS + COMP_RAW)
feature_cols = [c for c in train_df.columns if c not in drop]

print(f"Features: {len(feature_cols)}")

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

group_train = train_df.groupby("srch_id").size().values
group_val = val_df.groupby("srch_id").size().values

Train: 3,966,682  Val: 991,665


SVD features added: 43


Position-debiased features added: 6


Features: 136


In [4]:
params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5],
    "num_leaves": 127,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "min_child_samples": 50,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
    **DEVICE_PARAMS,
}

train_set = lgb.Dataset(X_train, label=y_train, group=group_train)
val_set = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_set)

model = lgb.train(
    params,
    train_set,
    valid_sets=[val_set],
    num_boost_round=2000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

Training until validation scores don't improve for 50 rounds


[100]	valid_0's ndcg@5: 0.377792


[200]	valid_0's ndcg@5: 0.38473


[300]	valid_0's ndcg@5: 0.387227


[400]	valid_0's ndcg@5: 0.388105


[500]	valid_0's ndcg@5: 0.389073


Early stopping, best iteration is:
[497]	valid_0's ndcg@5: 0.38927


In [5]:
val_df_eval = val_df.copy()
val_df_eval["pred_score"] = model.predict(X_val)

baseline_ndcg = compute_ndcg_fast(val_df_eval, "prop_location_score1")
model_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")
print(f"Baseline NDCG@5 (location_score1): {baseline_ndcg:.4f}")
print(f"LambdaRank NDCG@5:                 {model_ndcg:.4f}")
print(f"Improvement:                        +{model_ndcg - baseline_ndcg:.4f}")

Baseline NDCG@5 (location_score1): 0.1727
LambdaRank NDCG@5:                 0.3900
Improvement:                        +0.2173


In [6]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print("Top 20 features by gain:")
print(importance.head(20).to_string(index=False))

Top 20 features by gain:
                         feature          gain
              hotel_ctr_unbiased 334533.850803
            prop_location_score2 232783.860485
             price_vs_historical 119758.315660
       prop_location_score2_rank 117421.351009
             dest_hotel_affinity  94638.262650
                  price_usd_rank  88567.209423
              svd_affinity_score  80551.001073
      hotel_book_rate_by_country  75320.967443
           price_usd_diff_median  65922.428421
       prop_starrating_diff_mean  63767.064173
                        score1d2  55169.261645
            prop_starrating_rank  53639.521438
        prop_location_score2_max  48609.123156
                   prop_id_count  47342.438069
             price_usd_diff_mean  43881.804297
                  promotion_flag  39679.925684
  prop_location_score2_diff_mean  39645.954981
     prop_review_score_diff_mean  36492.162062
          prop_review_score_rank  35183.389646
prop_location_score2_diff_median  3

## Hyperparameter Search

Grid search over key parameters. Evaluate each on validation NDCG@5.

In [7]:
search_space = {
    "num_leaves": [63, 127, 255],
    "learning_rate": [0.01, 0.05, 0.1],
    "min_child_samples": [20, 50, 100],
}

results = []
base_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5],
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "feature_pre_filter": False,
    "num_threads": -1,
    "verbose": -1,
    "seed": 42,
    **DEVICE_PARAMS,
}

keys = list(search_space.keys())
for combo in product(*search_space.values()):
    trial_params = {**base_params, **dict(zip(keys, combo))}

    train_set_hp = lgb.Dataset(X_train, label=y_train, group=group_train)
    val_set_hp = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_set_hp)

    m = lgb.train(
        trial_params,
        train_set_hp,
        valid_sets=[val_set_hp],
        num_boost_round=2000,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    )

    preds = m.predict(X_val)
    val_df_eval["pred_score"] = preds
    score = compute_ndcg_fast(val_df_eval, "pred_score")

    row = dict(zip(keys, combo))
    row["ndcg5"] = score
    row["n_trees"] = m.best_iteration
    results.append(row)
    print(f"  leaves={row['num_leaves']:>3}  lr={row['learning_rate']:.2f}  "
          f"min_child={row['min_child_samples']:>3}  -> NDCG@5={score:.4f}  trees={m.best_iteration}")

results_df = pd.DataFrame(results).sort_values("ndcg5", ascending=False)
print("\nTop 5 configurations:")
print(results_df.head().to_string(index=False))

  leaves= 63  lr=0.01  min_child= 20  -> NDCG@5=0.3886  trees=1758


  leaves= 63  lr=0.01  min_child= 50  -> NDCG@5=0.3880  trees=1460


  leaves= 63  lr=0.01  min_child=100  -> NDCG@5=0.3898  trees=1757


  leaves= 63  lr=0.05  min_child= 20  -> NDCG@5=0.3895  trees=537


  leaves= 63  lr=0.05  min_child= 50  -> NDCG@5=0.3894  trees=549


...

  leaves=255  lr=0.05  min_child= 50  -> NDCG@5=0.3905  trees=253


  leaves=255  lr=0.05  min_child=100  -> NDCG@5=0.3898  trees=282


  leaves=255  lr=0.10  min_child= 20  -> NDCG@5=0.3855  trees=151


  leaves=255  lr=0.10  min_child= 50  -> NDCG@5=0.3858  trees=139


  leaves=255  lr=0.10  min_child=100  -> NDCG@5=0.3862  trees=167

Top 5 configurations:
 num_leaves  learning_rate  min_child_samples    ndcg5  n_trees
        255           0.01                100 0.392405     1210
        255           0.01                 20 0.391782     1089
        127           0.01                 50 0.391226     1638
        127           0.01                 20 0.390529     1418
        255           0.05                 50 0.390466      253


## Retrain with Best Params

Use the best configuration from the grid search, then save the model.

In [8]:
best = results_df.iloc[0]
print(f"Best config: leaves={int(best.num_leaves)}, lr={best.learning_rate}, "
      f"min_child={int(best.min_child_samples)}, NDCG@5={best.ndcg5:.4f}")

best_params = {
    **base_params,
    "num_leaves": int(best.num_leaves),
    "learning_rate": best.learning_rate,
    "min_child_samples": int(best.min_child_samples),
}

train_set_best = lgb.Dataset(X_train, label=y_train, group=group_train)
val_set_best = lgb.Dataset(X_val, label=y_val, group=group_val, reference=train_set_best)

best_model = lgb.train(
    best_params,
    train_set_best,
    valid_sets=[val_set_best],
    num_boost_round=2000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

val_df_eval["pred_score"] = best_model.predict(X_val)
final_ndcg = compute_ndcg_fast(val_df_eval, "pred_score")
print(f"\nFinal NDCG@5: {final_ndcg:.4f}")

models_dir = REPO_ROOT / "models"
models_dir.mkdir(exist_ok=True)
best_model.save_model(str(models_dir / "lambdarank_best.txt"))

Best config: leaves=255, lr=0.01, min_child=100, NDCG@5=0.3924


Training until validation scores don't improve for 50 rounds


[100]	valid_0's ndcg@5: 0.368971


[200]	valid_0's ndcg@5: 0.374193


[300]	valid_0's ndcg@5: 0.377549


...

[700]	valid_0's ndcg@5: 0.386904


[800]	valid_0's ndcg@5: 0.388711


[900]	valid_0's ndcg@5: 0.389703


Early stopping, best iteration is:
[947]	valid_0's ndcg@5: 0.390433



Final NDCG@5: 0.3911


In [9]:
out_dir = REPO_ROOT / "outputs"
out_dir.mkdir(exist_ok=True)

results_out = {
    "model": "lambdarank",
    "use_sample": USE_SAMPLE,
    "sample_frac": SAMPLE_FRAC if USE_SAMPLE else None,
    "n_rows": len(df),
    "n_searches": int(df["srch_id"].nunique()),
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_features": len(feature_cols),
    "n_trees": int(best_model.best_iteration),
    "best_params": {k: v for k, v in best_params.items() if k not in ("verbose", "num_threads", "seed")},
    "baseline_ndcg5": round(baseline_ndcg, 4),
    "model_ndcg5": round(final_ndcg, 4),
    "top_features": {row["feature"]: round(row["gain"], 1) for _, row in importance.head(15).iterrows()},
}

with open(out_dir / "lambdarank_run1.json", "w") as f:
    json.dump(results_out, f, indent=2)
print("Results saved to outputs/lambdarank_run1.json")

val_preds = val_df_eval[["srch_id", "prop_id", "booking_bool", "click_bool", "pred_score"]].copy()
val_preds.to_parquet(out_dir / "lambdarank_val_preds.parquet", index=False)
print(f"Validation predictions saved: {len(val_preds):,} rows")

Results saved to outputs/lambdarank_run1.json
Validation predictions saved: 991,665 rows
